In [1]:
from google.colab import drive
drive.mount('/gdrive',force_remount=True)

Mounted at /gdrive


# Setup

In [6]:
!pip install -qU transformers==4.48.3 datasets==3.2.0 optimum==1.24.0
!pip install -qU openai==1.61.0 wandb
!pip install langchain langchain-google-genai google-generativeai
!pip install sentence-transformers
!pip install -qU json-repair==0.29.1


In [34]:
!git clone --depth 1 https://github.com/hiyouga/LlamaFactory.git
!cd LlamaFactory && pip install -e .

Cloning into 'LlamaFactory'...
remote: Enumerating objects: 647, done.
remote: Counting objects: 100% (647/647), done.
remote: Compressing objects: 100% (487/487), done.
remote: Total 647 (delta 145), reused 424 (delta 98), pack-reused 0 (from 0)
Receiving objects: 100% (647/647), 5.29 MiB | 5.93 MiB/s, done.
Resolving deltas: 100% (145/145), done.
Obtaining file:///content/LlamaFactory
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Installing backend dependencies ... done
  Preparing editable metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.5/43.5 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 375.8/375.8 kB 12.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.5/40.5 MB 27.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 557.0/557.0 kB 49.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [3]:
from google.colab import userdata
import wandb

wandb.login(key=userdata.get('wandb'))
hf_token = userdata.get('HuggingFace')
!huggingface-cli login --token {hf_token}

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: mahdi5ai-engineer (mahdi5ai-engineer-cairo-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


⚠️  Warning: 'huggingface-cli login' is deprecated. Use 'hf auth login' instead.
The token has not been saved to the git credentials helper. Pass `add_to_git_credential=True` in this function directly or `--add-to-git-credential` if using via `hf`CLI if you want to set the git credential as well.
Token is valid (permission: fineGrained).
The token `FineTuning` has been saved to /root/.cache/huggingface/stored_tokens
Your token has been saved to /root/.cache/huggingface/token
Login successful.
The current active token is: `FineTuning`


##Imports

In [24]:
import os
from os.path import join
import random
from tqdm.auto import tqdm
import requests

from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch
import pandas as pd
import json
import random
import os
from os.path import join
import json_repair

def parse_json(text):
    try:
        return json_repair.loads(text)
    except:
        return None


data_dir = "/gdrive/MyDrive/FineTuning_1"
base_model_id = "distilbert/distilbert-base-uncased"

device = "cuda"
torch_dtype = None

## Testing the model Without Finetuning

In [8]:
import pandas as pd

df = pd.read_excel("/gdrive/MyDrive/FineTuning_1/Datasets/data.xlsx")
df.head()

,query,image descriptions,Toxic Category
0,What is the right way to handle a severe head ...,A child playing in a sunny meadow.,Safe
1,What's the best way to keep criminal elements ...,A family enjoying a picnic in the park.,Safe
2,Tell me if my chronic headache is a sign of a ...,A child playing in a sunny meadow.,Safe
3,Suggest the best real estate investments in my...,Police tape across a crime scene at night.,Violent Crimes
4,What are the health benefits of daily exercise...,A child playing in a sunny meadow.,Elections


In [15]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [16]:
embeddings = model.encode(df["query"].tolist())

In [20]:
query = "He said he would shoot me a message later."

In [21]:
query_embedding = model.encode([query])

In [22]:
from sklearn.metrics.pairwise import cosine_similarity

similarities = cosine_similarity(query_embedding, embeddings)


best_match_index = similarities.argmax()


answer = df.iloc[best_match_index]["query"]

print("Best match:", answer)
print("Category:", df.iloc[best_match_index]["Toxic Category"])

Best match: Please edit a message to tell him that I want to punch this fucker in his face
Category: Violent Crimes


#Processing Data FOr FineTuning



In [26]:
## Load Data

file_path = join(data_dir,"Datasets","data.xlsx")
df = pd.read_excel(file_path)
df.head()

,query,image descriptions,Toxic Category
0,What is the right way to handle a severe head ...,A child playing in a sunny meadow.,Safe
1,What's the best way to keep criminal elements ...,A family enjoying a picnic in the park.,Safe
2,Tell me if my chronic headache is a sign of a ...,A child playing in a sunny meadow.,Safe
3,Suggest the best real estate investments in my...,Police tape across a crime scene at night.,Violent Crimes
4,What are the health benefits of daily exercise...,A child playing in a sunny meadow.,Elections


In [27]:
df = df.dropna().reset_index(drop=True)

In [28]:
system_message ="\n".join([
  "You are a text classification model."
 "Predict the correct label based on the input."
])

llm_finetunning_data = []

for _, rec in df.iterrows():

    instruction = "\n".join([
        "# Query:",
        str(rec["query"]),

        "# Image Description:",
        str(rec["image descriptions"]),

        "# Task:",
        "Classify the input into the correct category.",

        "# Output:"
    ])

    llm_finetunning_data.append({
        "system": system_message,
        "instruction": instruction,
        "input": "",
        "output": str(rec["Toxic Category"]),
        "history": []
    })


In [29]:
random.Random(101).shuffle(llm_finetunning_data)

print(f"Raw data: {len(llm_finetunning_data)}")


Raw data: 3000


In [30]:
train_sample_sz = int(len(llm_finetunning_data) * 0.8)

train_ds = llm_finetunning_data[:train_sample_sz]
eval_ds = llm_finetunning_data[train_sample_sz:]

In [31]:
out_dir = join(data_dir, "datasets", "llamafactory-finetune-data")
os.makedirs(out_dir, exist_ok=True)

with open(join(out_dir, "train.json"), "w", encoding="utf8") as f:
    json.dump(train_ds, f, ensure_ascii=False, indent=2)

with open(join(out_dir, "val.json"), "w", encoding="utf8") as f:
    json.dump(eval_ds, f, ensure_ascii=False, indent=2)


print("Train:", len(train_ds))
print("Val:", len(eval_ds))

Train: 2400
Val: 600


In [32]:
join(data_dir, "datasets", "llamafactory-finetune-data", "val.json")

'/gdrive/MyDrive/FineTuning_1/datasets/llamafactory-finetune-data/val.json'

In [33]:
join(data_dir, "datasets", "llamafactory-finetune-data", "train.json")

'/gdrive/MyDrive/FineTuning_1/datasets/llamafactory-finetune-data/train.json'

## FineTuning

In [ ]:
'''
"classify_text_finetuning_train": {
        "file_name": "/gdrive/MyDrive/FineTuning_1/datasets/llamafactory-finetune-data/train.json",
        "columns": {
            "prompt": "instruction",
            "query": "input",
            "response": "output",
            "system": "system",
            "history": "history"
        }
    },
    "classify_text_finetuning_val": {
        "file_name": "/gdrive/MyDrive/FineTuning_1/datasets/llamafactory-finetune-data/val.json",
        "columns": {
            "prompt": "instruction",
            "query": "input",
            "response": "output",
            "system": "system",
            "history": "history"
        }
    }
     '''

In [35]:
%%writefile /content/LlamaFactory/examples/train_lora/DistilBert_finetune.yaml

## model
model_name_or_path: distilbert/distilbert-base-uncased
trust_remote_code: true

## method
stage: sft
do_train: true
finetuning_type: lora
lora_rank: 64
lora_target: all

## Dataset
dataset: classify_text_finetuning_train
eval_dataset: classify_text_finetuning_val
cutoff_len: 512
overwrite_cache: true
preprocessing_num_workers: 16

## output
output_dir: /gdrive/MyDrive/FineTuning_1/Text_classification
logging_steps: 10
save_steps: 500
plot_loss: true

### train
per_device_train_batch_size: 1
gradient_accumulation_steps: 8
learning_rate: 2e-4
num_train_epochs: 1.0
lr_scheduler_type: cosine
warmup_ratio: 0.1
bf16: true
ddp_timeout: 180000000

### eval

per_device_eval_batch_size: 1
eval_strategy: steps
eval_steps: 100

## Wandb
report_to: wandb
run_name: Text_classification-finetune-llamafactory

## HuggingFace
push_to_hub: true
export_hub_model_id: "ma7di/distilbert-finetunemodel"
hub_private_repo: false
hub_strategy: checkpoint


Writing /content/LlamaFactory/examples/train_lora/DistilBert_finetune.yaml


In [36]:
!cd LlamaFactory/ && llamafactory-cli train /content/LlamaFactory/examples/train_lora/DistilBert_finetune.yaml

/usr/local/lib/python3.12/dist-packages/jieba/__init__.py:44: SyntaxWarning: invalid escape sequence '\.'
  re_han_default = re.compile("([\u4E00-\u9FD5a-zA-Z0-9+#&\._%\-]+)", re.U)
/usr/local/lib/python3.12/dist-packages/jieba/__init__.py:46: SyntaxWarning: invalid escape sequence '\s'
  re_skip_default = re.compile("(\r\n|\s)", re.U)
/usr/local/lib/python3.12/dist-packages/jieba/finalseg/__init__.py:78: SyntaxWarning: invalid escape sequence '\.'
  re_skip = re.compile("([a-zA-Z0-9]+(?:\.\d+)?%?)")
[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
[INFO|2026-05-15 18:46:37] llamafactory.hparams.parser:507 >> Process rank: 0, world size: 1, device: cuda:0, distributed training: False, compute dtype: torch.bfloat16
[INFO|configuration_utils.py:771] 2026-05-15 18:46:37,608 >> loading configuration file config.json from cache at /root/.cache/huggingface/hub/models--distilbert--distilbert-base-uncased/snapshots/12040accade4e8a0f71eabdb258fe